# Job quickstart

Same recipe as `examples/job_quickstart.py`: one `nvidia-smi` job, `gpu="h100"`, timeout `1h`. Uses the active Nebius profile.

Jupyter already runs an event loop, so each live call is the awaited twin of the script. A synchronous `create()`, `get()`, `wait()`, `cancel()`, or `jobs.get()` in this kernel raises `ValidationError` and names that twin.

Run the cells in order. The dry-run cell resolves the project and region and does not create a job. The create cell does.

In [ ]:
import nebius_serverless as ns

COMMAND = ["nvidia-smi"]
IMAGE = "nvidia/cuda:13.1.1-runtime-ubuntu24.04"
GPU = "h100"
TIMEOUT = "1h"

## Recipe

Constructing a `Job` does not contact Nebius. The name is generated when you leave it unset.

In [ ]:
job = ns.Job(COMMAND, image=IMAGE, gpu=GPU, timeout=TIMEOUT)
{
    "name": job.name,
    "command": list(job.command),
    "image": job.image,
    "gpu": job.gpu,
    "timeout_seconds": job.timeout_seconds,
}

## Dry run

Resolves the project and region, returns the create request, and does not create a job.

In [ ]:
await job.create_async(dry_run=True)

## Create

Returns a `Run` once the id is known. The container has not finished.

In [ ]:
run = await job.create_async()
saved_id = run.id
saved_id

## Get

One fetch of the job JSON.

In [ ]:
await run.get_async()

## Wait

Polls until `status.state` is `COMPLETED`, `FAILED`, `CANCELLED`, or `ERROR`, then returns that record. `nvidia-smi` should reach `COMPLETED`.

In [ ]:
await run.wait_async()

## Cancel

Returns when the cancel request is accepted. The payload does not claim the container has stopped.

In [ ]:
await run.cancel_async()

## Reload the saved id

`jobs.get_async` fetches that id and does not create a job.

In [ ]:
loaded = await ns.jobs.get_async(saved_id)
{
    "id": loaded.id,
    "name": loaded.name,
    "status": loaded.status,
    "record": loaded.record,
}